<a href="https://colab.research.google.com/github/kaiju-no-9/Pytorch-Notes-/blob/main/class_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Class 8: Build a Recursive Language Model (RLM) from Scratch

By the end of this notebook, you will have:
1. **Seen long-context failure** in action — a model failing on a simple task with too much context
2. **Built an RLM from scratch** — the REPL, the agent loop, and recursive sub-LLM calls
3. **Compared vanilla LLM vs RLM** on the same task

Based on the paper: [Recursive Language Models](https://arxiv.org/abs/2512.24601) by Alex Zhang, Tim Kraska & Omar Khattab (MIT CSAIL)

---

## Part 0: Setup

We use OpenRouter for free model access, same as the agents class.

In [ ]:
import requests
import json
import re
import io
import sys
import traceback
from contextlib import redirect_stdout, redirect_stderr
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
# Models — use cheap/free ones for the class
ROOT_MODEL = "google/gemini-3-flash-preview"  # root agent (stronger)
SUB_MODEL  = "google/gemini-3-flash-preview"  # sub-agent (can be cheaper)

def llm_call(prompt, system="", model=ROOT_MODEL, max_tokens=4000):
    """Call an LLM via OpenRouter. Returns the text response."""
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})

    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={"model": model, "messages": msgs, "max_tokens": max_tokens}
    )
    data = r.json()
    if "choices" not in data:
        raise Exception(f"API error: {data}")
    return data["choices"][0]["message"]["content"]

# Quick test
print(llm_call("Say hello in one word."))

Hello.


---
## Part 1: The Problem — Long Context Fails

Let's create a task that's easy for humans but hard for LLMs at scale:
**Count specific items buried in a long, noisy dataset.**

In [ ]:
import random
random.seed(42)

# ──────────────────────────────────────────────
# Generate a synthetic dataset: people with cities and professions
# ──────────────────────────────────────────────

first_names = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank",
               "Grace", "Hank", "Ivy", "Jack", "Karen", "Leo",
               "Mona", "Nate", "Olivia", "Paul", "Quinn", "Rita",
               "Sam", "Tina", "Uma", "Vince", "Wendy", "Xander", "Yara", "Zane"]

cities = ["New York", "London", "Tokyo", "Paris", "Berlin",
          "Mumbai", "Sydney", "Toronto", "Dubai", "Singapore",
          "Seoul", "Bangkok", "Cairo", "Lagos", "Lima"]

professions = ["engineer", "doctor", "teacher", "artist", "chef",
               "pilot", "lawyer", "nurse", "writer", "musician"]

def generate_dataset(n_entries=500):
    """Generate a dataset of people with names, cities, and professions."""
    entries = []
    for i in range(n_entries):
        name = f"{random.choice(first_names)} {random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')}."
        city = random.choice(cities)
        prof = random.choice(professions)
        age = random.randint(22, 65)
        entries.append(f"Entry {i+1}: {name}, age {age}, {prof}, based in {city}")
    return entries

entries = generate_dataset(5000)
dataset_text = "\n".join(entries)

# Ground truth: count engineers in Tokyo
ground_truth = sum(1 for e in entries if "engineer" in e and "Tokyo" in e)

print(f"Dataset: {len(entries)} entries, {len(dataset_text)} characters")
print(f"\nFirst 5 entries:")
for e in entries[:5]:
    print(f"  {e}")
print(f"\n🎯 Ground truth: {ground_truth} engineers in Tokyo")

Dataset: 5000 entries, 265745 characters

First 5 entries:
  Entry 1: Uma D., age 37, chef, based in New York
  Entry 2: Hank E., age 65, doctor, based in Bangkok
  Entry 3: Xander R., age 49, musician, based in London
  Entry 4: Bob A., age 36, artist, based in London
  Entry 5: Quinn T., age 34, writer, based in New York

🎯 Ground truth: 40 engineers in Tokyo


### Try it with a vanilla LLM — just stuff the context in

In [ ]:
# ──────────────────────────────────────────────
# VANILLA APPROACH: stuff everything into the prompt
# ──────────────────────────────────────────────

vanilla_prompt = f"""Here is a dataset of people. Count EXACTLY how many are engineers in Tokyo.
Return ONLY the number, nothing else.

{dataset_text}"""

print(f"Prompt length: {len(vanilla_prompt)} characters")
print("Asking vanilla LLM...")

vanilla_answer = llm_call(vanilla_prompt)
print(f"\nVanilla LLM answer: {vanilla_answer}")
print(f"Ground truth:        {ground_truth}")
print(f"Correct?             {'✅ Yes' if str(ground_truth) in vanilla_answer else '❌ No'}")

Prompt length: 265860 characters
Asking vanilla LLM...

Vanilla LLM answer: 12
Ground truth:        40
Correct?             ❌ No


Even if the model gets this one right with 500 entries, the approach doesn't scale.
At 5,000 or 50,000 entries, context rot destroys accuracy.

**The RLM approach: instead of feeding all 500 entries to the model, let the model write code to count them itself.**

---
## Part 2: Build the REPL Environment

The REPL is where the model's code runs. We need:
- The context stored as a variable (not in the prompt)
- `print()` output captured and returned to the model
- A `FINAL()` function to signal the answer
- A `llm_query()` function for recursive sub-LLM calls

In [ ]:
# ──────────────────────────────────────────────
# THE REPL: a Python execution environment
# ──────────────────────────────────────────────

class RLMRepl:
    """A REPL environment for RLM.

    The context is stored as a Python variable.
    The model writes code that runs here.
    print() output is captured and returned to the model.
    """

    def __init__(self, context: str, max_output_chars: int = 5000):
        self.final_answer = None
        self.max_output_chars = max_output_chars
        self.sub_call_count = 0

        # The namespace where the model's code runs.
        # 'context' is the key variable — the long input stored here,
        # NOT in the LLM's prompt.
        self.namespace = {
            "context": context,            # <-- the long input lives here
            "FINAL": self._final,           # call FINAL("answer") to finish
            "llm_query": self._llm_query,   # call a sub-LLM (recursion!)
            # Standard library modules the model might want
            "re": re,
            "json": json,
            "len": len,
            "print": print,
            "int": int,
            "float": float,
            "str": str,
            "list": list,
            "dict": dict,
            "range": range,
            "enumerate": enumerate,
            "sum": sum,
            "sorted": sorted,
            "min": min,
            "max": max,
            "abs": abs,
            "set": set,
            "tuple": tuple,
            "zip": zip,
            "map": map,
            "filter": filter,
            "isinstance": isinstance,
            "type": type,
            "True": True,
            "False": False,
            "None": None,
        }

    def _final(self, answer):
        """Called by the model to submit its final answer."""
        self.final_answer = str(answer)
        print(f"[FINAL ANSWER SUBMITTED: {answer}]")

    def _llm_query(self, query: str, sub_context: str = "") -> str:
        """Recursive sub-LLM call.

        The model can call this from within its code to get
        a sub-LLM to process a chunk of context.
        The result comes back as a string variable — NOT
        loaded into the parent's context window.
        """
        self.sub_call_count += 1
        print(f"  [Sub-LLM call #{self.sub_call_count}: '{query[:80]}...'")

        prompt = query
        if sub_context:
            prompt = f"{query}\n\nContext:\n{sub_context}"

        result = llm_call(prompt, model=SUB_MODEL, max_tokens=2000)
        print(f"   Sub-LLM returned: '{result[:100]}...']")
        return result

    def execute(self, code: str) -> str:
        """Execute Python code in the REPL. Returns captured stdout."""
        stdout_capture = io.StringIO()
        stderr_capture = io.StringIO()

        try:
            with redirect_stdout(stdout_capture), redirect_stderr(stderr_capture):
                exec(code, self.namespace)
            output = stdout_capture.getvalue()
        except Exception as e:
            output = f"ERROR: {type(e).__name__}: {e}"

        # Truncate if too long — the model shouldn't be overwhelmed
        if len(output) > self.max_output_chars:
            output = output[:self.max_output_chars] + f"\n... [truncated to {self.max_output_chars} chars]"

        return output

# Quick test of the REPL
repl = RLMRepl(context="Hello world! This is a test context with some data.")
print("Test 1 — peek at context:")
print(repl.execute('print(context[:30])'))

print("\nTest 2 — search with regex:")
print(repl.execute('import re\nmatches = re.findall(r"\\w+", context)\nprint(f"Words: {len(matches)}")'))

print("\nTest 3 — submit final answer:")
print(repl.execute('FINAL(42)'))
print(f"Final answer stored: {repl.final_answer}")

print("\n✅ REPL works!")

Test 1 — peek at context:
Hello world! This is a test co


Test 2 — search with regex:
Words: 10


Test 3 — submit final answer:
[FINAL ANSWER SUBMITTED: 42]

Final answer stored: 42

✅ REPL works!


---
## Part 3: Build the RLM Agent Loop

The core loop:
1. Give the LLM the query + system prompt (but NOT the context)
2. LLM writes Python code
3. Execute the code in the REPL (where context lives)
4. Send the output back to the LLM
5. Repeat until FINAL() is called or we hit max iterations

In [ ]:
# ──────────────────────────────────────────────
# THE RLM SYSTEM PROMPT
# This tells the model HOW to use the REPL
# ──────────────────────────────────────────────

RLM_SYSTEM_PROMPT = """You are an RLM (Recursive Language Model) agent.

You have access to a Python REPL environment. The user's data is stored
in a variable called `context` — it may be very long (millions of characters).
You CANNOT see the context directly. You must write Python code to explore it.

Available tools:
- `context` — the full input text (Python string variable)
- `print()` — use this to see output from your code
- `llm_query(query, sub_context)` — call a sub-LLM to analyze a chunk.
  The sub-LLM's response is returned as a string. It does NOT enter your context.
- `FINAL(answer)` — call this when you have the final answer.
- Standard Python: `re`, `json`, `len`, `sum`, etc.

Strategy:
1. First, check the size: `print(len(context))`
2. Peek at the structure: `print(context[:500])`
3. Use code to search, filter, count, or slice the data
4. For complex subtasks, use `llm_query()` to delegate to a sub-LLM
5. When done, call `FINAL(your_answer)`

Rules:
- Write ONLY Python code. No markdown, no explanation.
- Your code block must be wrapped in ```python ... ```
- Use print() to see results — you only see what you print.
- Variables persist between steps (like Jupyter cells).
- Be systematic. Explore first, then solve.
"""

print("System prompt defined ✅")

System prompt defined ✅


In [ ]:
# ──────────────────────────────────────────────
# THE RLM LOOP
# ──────────────────────────────────────────────

def extract_code(response: str) -> str:
    """Extract Python code from the LLM's response."""
    # Try to find ```python ... ``` blocks
    pattern = r'```python\s*\n(.*?)```'
    matches = re.findall(pattern, response, re.DOTALL)
    if matches:
        return matches[0].strip()

    # Try ``` ... ``` blocks
    pattern = r'```\s*\n(.*?)```'
    matches = re.findall(pattern, response, re.DOTALL)
    if matches:
        return matches[0].strip()

    # If no code blocks, try to use the whole response as code
    # (strip any leading text before first line that looks like code)
    lines = response.strip().split('\n')
    code_lines = []
    started = False
    for line in lines:
        if not started and line.startswith(('import ', 'from ', 'print(', '#', 'for ', 'if ', 'def ', 'context', 'result', 'count', 'data', 'lines', 'FINAL')):
            started = True
        if started:
            code_lines.append(line)

    return '\n'.join(code_lines) if code_lines else response.strip()


def run_rlm(query: str, context: str, max_iterations: int = 10, verbose: bool = True):
    """Run an RLM agent on a query with a given context.

    The context is NOT sent to the LLM. It's stored in the REPL
    as a Python variable. The LLM writes code to explore it.

    Args:
        query: The question to answer
        context: The (potentially huge) input text
        max_iterations: Max REPL interaction loops
        verbose: Print each step

    Returns:
        dict with 'answer', 'iterations', 'sub_calls', 'history'
    """
    repl = RLMRepl(context)
    history = []  # conversation history for the root LLM

    # First message: just the query (NOT the context!)
    user_msg = f"""Task: {query}

The data is in the `context` variable ({len(context)} characters long).
Write Python code to explore and solve this. Start by checking the structure."""

    history.append({"role": "user", "content": user_msg})

    for i in range(max_iterations):
        if verbose:
            print(f"\n{'='*60}")
            print(f"  ITERATION {i+1}/{max_iterations}")
            print(f"{'='*60}")

        # Ask the LLM to write code
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
            json={
                "model": ROOT_MODEL,
                "messages": [{"role": "system", "content": RLM_SYSTEM_PROMPT}] + history,
                "max_tokens": 2000
            }
        ).json()

        assistant_msg = response["choices"][0]["message"]["content"]
        history.append({"role": "assistant", "content": assistant_msg})

        # Extract code from the response
        code = extract_code(assistant_msg)

        if verbose:
            print(f"\n📝 LLM wrote:\n")
            for line in code.split('\n'):
                print(f"    {line}")

        # Execute in the REPL
        output = repl.execute(code)

        if verbose:
            print(f"\n📤 REPL output:\n")
            for line in output.split('\n'):
                print(f"    {line}")

        # Check if FINAL was called
        if repl.final_answer is not None:
            if verbose:
                print(f"\n🏁 FINAL ANSWER: {repl.final_answer}")
                print(f"   Iterations: {i+1}")
                print(f"   Sub-LLM calls: {repl.sub_call_count}")
            return {
                "answer": repl.final_answer,
                "iterations": i + 1,
                "sub_calls": repl.sub_call_count,
                "history": history
            }

        # Feed the output back to the LLM for the next iteration
        history.append({
            "role": "user",
            "content": f"REPL output:\n```\n{output}\n```\nContinue. Write more code or call FINAL(answer) when done."
        })

    if verbose:
        print(f"\n⚠️ Max iterations reached without FINAL()")
    return {
        "answer": None,
        "iterations": max_iterations,
        "sub_calls": repl.sub_call_count,
        "history": history
    }

print("RLM agent loop defined ✅")

RLM agent loop defined ✅


---
## Part 4: Run the RLM!

Same task as before: count engineers in Tokyo.
But now the model writes code to do it — the context never enters the prompt.

In [ ]:
# ──────────────────────────────────────────────
# RLM APPROACH: context as variable, model writes code
# ──────────────────────────────────────────────

result = run_rlm(
    query="Count EXACTLY how many people are engineers based in Tokyo. Return just the number.",
    context=dataset_text,
    max_iterations=8,
    verbose=True
)

print(f"\n{'='*60}")
print(f"  COMPARISON")
print(f"{'='*60}")
print(f"RLM answer:     {result['answer']}")
print(f"Ground truth:   {ground_truth}")
print(f"Correct?        {'✅' if result['answer'] and str(ground_truth) in str(result['answer']) else '❌'}")
print(f"Iterations:     {result['iterations']}")
print(f"Sub-LLM calls:  {result['sub_calls']}")


  ITERATION 1/8

📝 LLM wrote:

    # First, let's explore the structure of the context to see how data is formatted.
    print(f"Context length: {len(context)}")
    print("Sample data:")
    print(context[:1000])

📤 REPL output:

    Context length: 265745
    Sample data:
    Entry 1: Uma D., age 37, chef, based in New York
    Entry 2: Hank E., age 65, doctor, based in Bangkok
    Entry 3: Xander R., age 49, musician, based in London
    Entry 4: Bob A., age 36, artist, based in London
    Entry 5: Quinn T., age 34, writer, based in New York
    Entry 6: Wendy U., age 48, writer, based in Bangkok
    Entry 7: Hank O., age 22, chef, based in Singapore
    Entry 8: Yara Z., age 43, lawyer, based in Tokyo
    Entry 9: Ivy E., age 28, pilot, based in Paris
    Entry 10: Charlie M., age 44, pilot, based in London
    Entry 11: Tina I., age 51, engineer, based in Cairo
    Entry 12: Rita D., age 27, lawyer, based in Lima
    Entry 13: Rita J., age 45, musician, based in Lagos
    Entry 1

### What just happened?

Watch the iterations above. The model likely:
1. Checked `len(context)` to see how big the data is
2. Peeked at `context[:500]` to understand the format
3. Wrote a regex or string filter to count matching entries
4. Called `FINAL(count)`

**The model never processed 500 entries through its attention mechanism.**
It wrote 3-4 lines of Python that processed them in milliseconds.

---
## Part 5: A Harder Task — Where Sub-LLMs Help

For counting, code alone was enough. But what about tasks that
require *understanding* — like summarizing or analyzing sentiment?

This is where `llm_query()` (recursive sub-LLM calls) shines.

In [ ]:
# ──────────────────────────────────────────────
# Generate a harder dataset: restaurant reviews
# ──────────────────────────────────────────────

review_templates = [
    "The {dish} was {quality}. Service was {service}. {extra}",
    "Ordered the {dish}. {quality} quality overall. {extra} Staff was {service}.",
    "Had the {dish} for lunch. {extra} Food was {quality}, service {service}.",
]

dishes = ["pasta", "sushi", "burger", "salad", "steak", "pizza", "tacos", "curry", "ramen", "risotto"]
qualities_good = ["excellent", "fantastic", "amazing", "superb", "delightful", "perfectly cooked"]
qualities_bad = ["terrible", "awful", "undercooked", "bland", "disappointing", "inedible"]
qualities_mid = ["okay", "decent", "average", "nothing special", "fine but forgettable"]
services_good = ["outstanding", "friendly and fast", "attentive", "excellent"]
services_bad = ["slow", "rude", "inattentive", "terrible"]
extras = ["Would definitely come back.", "Will not return.", "Overpriced.",
          "Great ambiance.", "Very noisy.", "Cozy atmosphere.", "Long wait times.",
          "Perfect date spot.", "Good for groups.", "Parking was easy.", ""]

restaurant_names = ["Bella Vita", "Sakura House", "The Grill Room", "Green Bowl",
                    "Chez Marie", "Spice Route", "Noodle Bar", "Ocean Plate",
                    "Fire & Stone", "The Garden Kitchen"]

def make_reviews(n=200):
    reviews = []
    for i in range(n):
        restaurant = random.choice(restaurant_names)
        dish = random.choice(dishes)
        sentiment = random.choices(["positive", "negative", "mixed"], weights=[0.5, 0.3, 0.2])[0]

        if sentiment == "positive":
            quality = random.choice(qualities_good)
            service = random.choice(services_good)
        elif sentiment == "negative":
            quality = random.choice(qualities_bad)
            service = random.choice(services_bad)
        else:
            quality = random.choice(qualities_mid)
            service = random.choice(random.choice([services_good, services_bad]))

        template = random.choice(review_templates)
        review = template.format(dish=dish, quality=quality, service=service, extra=random.choice(extras))
        reviews.append(f"Review {i+1} [{restaurant}]: {review}")
    return reviews

reviews = make_reviews(200)
reviews_text = "\n".join(reviews)

print(f"Reviews dataset: {len(reviews)} reviews, {len(reviews_text)} characters")
print(f"\nSample reviews:")
for r in reviews[:3]:
    print(f"  {r}")

Reviews dataset: 200 reviews, 20504 characters

Sample reviews:
  Review 1 [Spice Route]: Ordered the tacos. amazing quality overall. Great ambiance. Staff was outstanding.
  Review 2 [Ocean Plate]: The sushi was fantastic. Service was friendly and fast. Would definitely come back.
  Review 3 [Chez Marie]: Ordered the tacos. nothing special quality overall. Parking was easy. Staff was friendly and fast.


In [ ]:
# ──────────────────────────────────────────────
# SEMANTIC TASK: summarize the sentiment per restaurant
# This requires UNDERSTANDING, not just counting.
# The model should use llm_query() for sub-tasks.
# ──────────────────────────────────────────────

result2 = run_rlm(
    query="""Analyze these restaurant reviews. For each restaurant, provide:
1. How many reviews it has
2. Overall sentiment (positive/negative/mixed)
3. Most mentioned dish

Use code to group reviews by restaurant, then use llm_query() to analyze
the sentiment of each group. Return a summary.""",
    context=reviews_text,
    max_iterations=12,
    verbose=True
)

print(f"\n{'='*60}")
print(f"  FINAL RESULT")
print(f"{'='*60}")
print(f"Answer:\n{result2['answer']}")
print(f"\nIterations: {result2['iterations']}")
print(f"Sub-LLM calls: {result2['sub_calls']}")


  ITERATION 1/12

📝 LLM wrote:

    # First, let's explore the structure of the context.
    print(f"Total length: {len(context)}")
    print("First 1000 characters:")
    print(context[:1000])

📤 REPL output:

    Total length: 20504
    First 1000 characters:
    Review 1 [Spice Route]: Ordered the tacos. amazing quality overall. Great ambiance. Staff was outstanding.
    Review 2 [Ocean Plate]: The sushi was fantastic. Service was friendly and fast. Would definitely come back.
    Review 3 [Chez Marie]: Ordered the tacos. nothing special quality overall. Parking was easy. Staff was friendly and fast.
    Review 4 [Spice Route]: Ordered the salad. bland quality overall. Very noisy. Staff was terrible.
    Review 5 [Ocean Plate]: The pasta was superb. Service was excellent. Great ambiance.
    Review 6 [The Garden Kitchen]: Had the pizza for lunch.  Food was amazing, service excellent.
    Review 7 [Ocean Plate]: The steak was terrible. Service was terrible. Very noisy.
    Review 8 

### What happened here?

The model likely:
1. Peeked at the data structure
2. Used regex/code to **group reviews by restaurant name**
3. For each restaurant, used `llm_query()` to ask a sub-LLM to **analyze sentiment** of that group
4. Compiled the results

This is the key RLM pattern:
- **Code** handles what code is good at: parsing, grouping, counting, filtering
- **Sub-LLMs** handle what LLMs are good at: understanding, summarizing, analyzing
- **The root LLM** orchestrates the whole thing

Each sub-LLM only sees ONE restaurant's reviews — small, clean context. No context rot.




---
## Part 7: Scale Test — Does RLM Actually Scale Better?

Let's test with larger data to see the difference.

In [ ]:
# ──────────────────────────────────────────────
# SCALE TEST: 2000 entries
# ──────────────────────────────────────────────

big_entries = generate_dataset(20000)
big_dataset = "\n".join(big_entries)
big_truth = sum(1 for e in big_entries if "engineer" in e and "Tokyo" in e)

print(f"Big dataset: {len(big_entries)} entries, {len(big_dataset)} characters")
print(f"Ground truth: {big_truth} engineers in Tokyo")
print(f"\n--- Running RLM ---")

big_result = run_rlm(
    query="Count EXACTLY how many people are engineers based in Tokyo.",
    context=big_dataset,
    max_iterations=8,
    verbose=True
)

print(f"\n{'='*60}")
print(f"  SCALE TEST RESULTS")
print(f"{'='*60}")
print(f"Dataset size:   {len(big_entries)} entries")
print(f"RLM answer:     {big_result['answer']}")
print(f"Ground truth:   {big_truth}")
print(f"Correct?        {'✅' if big_result['answer'] and str(big_truth) in str(big_result['answer']) else '❌'}")
print(f"Iterations:     {big_result['iterations']}")
print(f"\nThe RLM processes 2000 entries without any of them")
print(f"entering the LLM's context window. The model wrote")
print(f"a few lines of Python that ran in milliseconds.")

Big dataset: 20000 entries, 1076295 characters
Ground truth: 146 engineers in Tokyo

--- Running RLM ---

  ITERATION 1/8

📝 LLM wrote:

    # First, let's explore the structure of the context to see how data is formatted.
    print(f"Context length: {len(context)}")
    print("First 1000 characters:")
    print(context[:1000])

📤 REPL output:

    Context length: 1076295
    First 1000 characters:
    Entry 1: Uma L., age 43, nurse, based in Berlin
    Entry 2: Xander E., age 26, artist, based in Lagos
    Entry 3: Diana E., age 50, teacher, based in Paris
    Entry 4: Jack Y., age 53, musician, based in Dubai
    Entry 5: Vince T., age 50, doctor, based in Bangkok
    Entry 6: Olivia Y., age 37, artist, based in Sydney
    Entry 7: Vince J., age 53, teacher, based in Seoul
    Entry 8: Frank A., age 56, writer, based in Paris
    Entry 9: Mona V., age 52, doctor, based in Tokyo
    Entry 10: Leo W., age 25, writer, based in New York
    Entry 11: Zane C., age 37, lawyer, based in Sin

---
## Part 8: Recap — What We Built

```
Traditional LLM:
  prompt (with ALL the data) → model → answer
  Problem: context rot, limited by window size

Our RLM:
  query (small) → root LLM → writes code → REPL executes it
      ↑                                        │
      └──── output ◄───────────────────────────┘
      
  context stored as variable in REPL (never in prompt)
  llm_query() for recursive sub-LLM calls
  FINAL() to submit the answer
```

### Paper results:
- RLM(GPT-5-mini) outperforms vanilla GPT-5 by 34+ points on long-context tasks
- Handles inputs 100x beyond model context windows  
- RLM-Qwen3-8B (trained with RL) improves 28.3% over base model

### Resources:
- **Paper**: [arxiv.org/abs/2512.24601](https://arxiv.org/abs/2512.24601)
- **Official code**: [github.com/alexzhang13/rlm](https://github.com/alexzhang13/rlm)
- **Prime Intellect blog**: [primeintellect.ai/blog/rlm](https://primeintellect.ai/blog/rlm)
- **Alex Zhang's blog**: [alexzhang13.github.io/blog/2025/rlm/](https://alexzhang13.github.io/blog/2025/rlm/)